In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from openai import OpenAI

def scrape_js_page(url: str, wait_css: str = "body", timeout: int = 20) -> str:
    """Load a JS-rendered page, wait for content, and return cleaned visible text."""
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1280,800")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)

        # Wait for something that appears after JS renders
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, wait_css))
        )

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        # Remove irrelevant tags
        for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
            tag.decompose()

        text = soup.get_text(separator="\n")

        # Cleanup
        lines = [ln.strip() for ln in text.splitlines()]
        lines = [ln for ln in lines if ln]
        return "\n".join(lines)
    finally:
        driver.quit()


In [ ]:
url = "https://edition.cnn.com/"
page_text = scrape_js_page(url, wait_css="body")

messages = [
    {"role":"system","content":"summarize the web text."},
    {"role":"user","content":page_text[:20000]}
    ]

print(page_text[:20000])

llm_used = OpenAI()
response = llm_used.chat.completions.create(model = "gpt-4.1-nano", messages = messages)
print(response.choices[0].message.content)